# Detailed Explanation of how this evaluation system works

## Terminology definition:
- **Problem**: The specific task or question that a model is being evaluated on. In this example, the problem is to generate a medical note based on a conversation between a doctor and a patient.
- **Generator**: The model that is responsible for generating the ouput based on the given input and prompt. In this example, the generator is a open-weights thinking LLM that takes in the input and generates a medical note.
- **System guidelines**: Each conversation comes with detailed instructions and rules of what and how the generator should generate. These guidelines are designed to ensure that the generated output is accurate, relevant, and adheres to specific requirements. See an example in `data/input.jsonl`.
- **Judge**: The model that is responsible for evaluating the quality of the generated output by generators based on specific criteria. In this example, the judge is a closed-weights LLM that assesses the generated medical note across multiple dimensions such as hallucination, adherence, and coherence.
- **Evaluation criteria**: The specific dimensions or aspects that the judge uses to evaluate the generated output. 
    In this example, the evaluation criteria include: 
        - hallucination (whether the generated note contains false information)
        - adherence (whether the generated note follows the system guidelines)
        - coherence (whether the generated note is logically consistent and well-structured)
    The judge is required to provide three separate scores for each of these criteria, along with a detailed explanation for each score.
- **Evaluator**: The prompt used to instruct the judge on how to evaluate the generated output. The evaluator provides specific instructions on how to assess the output based on the defined evaluation criteria. See an example in `src/evaluator.jsonl`.

## Workflow:
## First, some set up:


In [ ]:
# Setup
from src.api import *
# client = init_client()
client = None

# Define the model paths to be called by OpenAI API. You can add more models you want to use. 
MODELS = {
    "kimi": "moonshotai/kimi-k2.6", 
    "qwen3": "qwen/qwen3-235b-a22b-thinking-2507", 
    "deepseek": "deepseek/deepseek-r1-0528", 
    "gpt": "openai/gpt-5.5-20240924", 
    "gemini": "google/gemini-2.5-pro-20240924", 
    "claude": "anthropic/claude-3.5-20240924", # write your own proxy if you want to use Claude :)
}

# generator models to write notes. 
generators = {k: MODELS[k] for k in MODELS if k in ("kimi", "qwen3", "deepseek")}
# judge models to assess and evaluate the generated notes. We use stronger models as judges to ensure a more reliable evaluation.
judges = {k: MODELS[k] for k in MODELS if k in ("gpt", "gemini", "claude")}

We then ask generators to generate the responses to each input prompt following the same guidelines (system prompt). 

In [ ]:
inputs = read_jsonl("../data/input.jsonl")
resp_file = "../data/generation_resps.jsonl"
resps = read_jsonl(resp_file) if os.path.exists(resp_file) else []
existing_ids = {item["id"] for item in resps}
for item in [item for item in inputs if item["id"] not in existing_ids]:
    inp = generator_prompt(system_msg=item["system"], user_msg=item["user"])
    resps.append({
        "id": item["id"],
        "generator_resps": {model: safe_call(client, model_path, inp) for model, model_path in generators.items()}
    })
save_jsonl(resps, resp_file)

## Evaluator Design

### 1st Evaluator: Strict Grounding and Extractive Faithfulness

The first evaluator was derived primarily from **repeated patterns and constraints** embedded within the **prompt template** itself. Across the dataset, the user prompts appeared to be wrapped inside a large universal instruction template before being sent to the model, meaning the template likely encodes many of the customer’s actual preferences and deployment requirements. 

#### In this case, the workflow is not a general medical assistant or diagnostic agent, but rather a clinical documentation assistant whose purpose is to convert doctor-patient conversations into structured clinical notes suitable for medical records and clinician workflows. 

The prompt repeatedly emphasizes phrases such as **“do not infer”**, **“do not invent”**, **“only include if explicitly mentioned”**, and **“never provide unsupported diagnoses or findings”**, often multiple times throughout the instructions. Because these constraints are repeatedly reinforced in a healthcare setting where **factual reliability** and **medico-legal correctness** are critical, I believed that *strict extractive grounding* is likely the highest-priority evaluation axis. In other words, the model is expected to behave as a faithful clinical note taker rather than an autonomous medical reasoner. 

This introduces an intentional tradeoff: the evaluator **sacrifices** potential medical intelligence and *discourages the model from proposing plausible diagnoses or conditions that were not explicitly stated by the clinician*, even if they *could be reasonably inferred* from the symptoms. 

The goal is not to create a “smart doctor”, but a highly reliable and conservative documentation system that clinicians can trust not to silently modify or upgrade medical conclusions. 

#### At the same time, the evaluator still allows summarisation, paraphrasing, and restructuring of information into concise clinical terminology, since efficient clinical documentation naturally requires compression and standardisation of conversational language. 

However, these transformations must remain semantically equivalent to the original transcript and must not introduce new clinical meaning. Finally, besides preventing hallucinations, the evaluator also checks that clinically relevant information explicitly mentioned during the encounter is not omitted from the final note, ensuring the output remains both conservative and complete.

### 2nd Evaluator: Information Extraction Completeness and Clinical Recall

The second evaluator was derived from the observation that a clinically useful note must not only avoid hallucinations and follow formatting requirements, but also **retain the important medical information** discussed during the encounter. This is a complement to the first evaluator. 

A model could technically satisfy the other two evaluators by generating a conservative and well-structured note while still **omitting clinically meaningful details**, resulting in documentation that is **incomplete or less useful for downstream care**. From this, I believed that **information extraction completeness** should be treated as an independent evaluation axis focused on clinical recall rather than factual precision. The prompt itself repeatedly emphasizes capturing all relevant symptoms, history, investigations, treatments, and contextual information discussed during the consultation, suggesting that completeness is an important customer requirement alongside grounding and formatting. This evaluator therefore measures whether the generated note successfully captures and preserves **clinically relevant information** such as symptoms, relevant negatives, medication side effects, family history, investigation results, treatment discussions, and follow-up plans. 

The goal is not exhaustive transcription, but ensuring that important clinical information explicitly mentioned during the encounter is **retained in the final documentation**.

### 3rd Evaluator: Formatting Adherence

The third evaluator was derived from the observation that the prompt strongly emphizes not only on the correctness of the medical content, but also on the *exact structure, formatting, and stylistic presentation* of the generated note. 

Across the prompt, the user repeatedly specifies strict requirements regarding section ordering, **placeholder handling**, **bullet formatting**, **clinician writing style**, omission rules, date formatting, spelling conventions, and even phrasing preferences (eg. "Subjective", "Past medical history", "Objective", "Assessment & Plan"). This suggests that the generated note is intended to integrate directly into a real clinical documentation workflow or electronic medical record system, where formatting consistency and structural reliability are operational requirements rather than cosmetic preferences. From this, I inferred that formatting compliance should be evaluated as an independent quality axis separate from factual grounding. 

A note may contain entirely correct medical information while still failing due to placeholder leakage, incorrect section mapping, overly verbose prose, or failure to follow the clinician’s requested style and template hierarchy. The evaluator therefore focuses on whether the model can reliably execute a highly constrained documentation schema under long-context instruction-heavy settings. Importantly, this evaluator intentionally **does not** assess medical correctness or hallucination behaviour directly, but instead measures whether the model faithfully follows the requested documentation protocol and produces output that is structurally usable in downstream clinical workflows.

The responses from the judge should be the following json format, where the rationale should be a detailed explanation of why the judge gave the score for each criterion.

```python
judge_resp_temp = {
  "strict_grounding": {"rationale": "...", "score": 0},
  "information_completeness": {"rationale": "...", "score": 0},
  "formatting_adherence": {"rationale": "...", "score": 0}
}
```

In [ ]:
from src.evaluator import EVALUATORS
import numpy as np
import json

def judge_and_eval(judge_model_path, input_item, gen_resp):
    eval_results = {}
    for evaluator in EVALUATORS:
        eval_prompt = evaluation_prompt(evaluator, input_item, gen_resp)
        eval_resp = safe_call(client, judge_model_path, eval_prompt)
        try:
            eval_resp = json.loads(eval_resp)
        except json.JSONDecodeError:
            prRed(f"Failed to parse evaluation response from the judge model. Response: {eval_resp}")
            eval_resp = {k: None for k in evaluator.keys()}  # Set all evaluation dimensions to None if parsing fails
        eval_results[evaluator] = eval_resp
    return eval_results

def retrieve_score(eval_results):
    """Helper function to retrieve the scores (digits) from the evaluation results, and return a list of scores corresponding to the evaluation dimensions."""
    return [eval_results[evaluator]["score"] if evaluator in eval_results else None for evaluator in EVALUATORS]

inputs = read_jsonl("../data/input.jsonl")
resps = read_jsonl("../data/generation_resps.jsonl")
# use a 4D array to store all eval results: (input_id, judge_model_id, generator_model_id, evaluator_id), don't mix up the dimensions :)
results = np.load("../data/results.npy") if os.path.exists("../data/results.npy") else np.zeros((len(resps), len(judges), len(generators), len(EVALUATORS)))
for judge_id, (judge_model, judge_model_path) in enumerate(judges.items()):
    judge_resp_file = f"../data/judge_resps_{judge_model}.jsonl"
    judge_resps = read_jsonl(judge_resp_file) if os.path.exists(judge_resp_file) else []
    existing_ids = {item["id"] for item in judge_resps}
    for item in [res for res in resps if res["id"] not in existing_ids]:
        idx = int(item["id"])
        input_item = next((i for i in inputs if i["id"] == idx), None)
        inp = generator_prompt(system_msg=input_item["system"], user_msg=input_item["user"])
        for generator_id, (model, gen_resp) in enumerate(item["generator_resps"].items()):
            eval_resp = judge_and_eval(judge_model_path, input_item, gen_resp)
            judge_resps.append({
                "id": idx,
                "generator_model": model,
                "judge_model": judge_model,
                "evaluation": eval_resp
            })
            results[idx, judge_id, generator_id, :] = retrieve_score(eval_resp)
    save_jsonl(judge_resps, judge_resp_file)
np.save("../data/results.npy", results)

After collecting all evaluation results, you can analyze the results array to compare the performance of different generator models under different judges and evaluators. 

## Selecting the Best Judge
The key insight: a good judge is one whose verdicts are consistent with the consensus, but also discriminating (not always 0 or 1). Three complementary signals:
- Signal 1 — Consensus Alignment
Without ground truth, the best proxy for "correct" is the majority vote across all judges. A good judge agrees with that consensus more than a bad one. This is the "wisdom of crowds" assumption.
- Signal 2 — Discrimination Power
A judge that outputs 95% ones is useless even if it "agrees" with other lenient judges. Measure how well a judge separates generators — a good judge should score good generators higher than bad ones, consistently.
- Signal 3 — Self-consistency
Across repeated or similar items, a good judge should be stable. We already built this in v2 as cross-evaluator std.
- Combining: Judge Score
```python
judge_score = α·consensus_alignment - β·bias_penalty - γ·inconsistency
```

## Selecting the Best Generator

The key design decisions, and why
Judge ranking — why these 4 signals?
Each catches a different failure mode:
- alignment:A rogue judge who consistently disagrees with everyone
- discrimination: A useless judge who scores everything ~0.7 with no spread
- consistency: An unstable judge whose verdicts vary arbitrarily across eval dims
- bias_penalty: A lenient/strict judge who inflates or deflates scores systematically

Generator ranking — why three aggregation strategies?
- mean_scores: scores highest on average
    - use when all eval dims matter roughly equally
- weighted_eval_scores: scores highest on your priorities
    - use when certain eval dims are more important than others
- min_score (weakest-link): is most balanced — no terrible dimension
    - use when you can't tolerate any single failure mode

In [ ]:
from src.judge_agreement_criteria import stratified_report
from src.judge_and_gen_selection import selection_report

metrics = stratified_report(results, judge_names=judges , input_names=inputs.keys(), gen_names=generators.keys(), eval_names=EVALUATORS.keys())

eval_weights = np.array([2.0, 1.0, 1.0])
selection_metrics = selection_report(results, judge_names=judges , input_names=inputs.keys(), gen_names=generators.keys(), eval_names=EVALUATORS.keys(), eval_weights=eval_weights)

## Sample outputs

Warning: The "wisdom of crowds" assumption in consensus alignment only holds if judges are independently noisy, not systematically wrong together. If all your closed-weight LLMs share the same training biases, consensus may just amplify that shared bias. The ground truth verification block (simulation-only) lets you audit this when you have labels.

The following is the sample output. Note that we have wrapped everything in a single object `scorer` for easier usage, so you don't actually need to run anything above. Refer to `README.md` for how to use the scorer object in your own evaluation pipeline.

```python
"""
================================================================
  JUDGE & GENERATOR SELECTION REPORT
================================================================

────────────────────────────────────────────────
PART 1 — JUDGE RANKING
────────────────────────────────────────────────

  Signals (all normalized to [0,1] per judge before combining):
  ┌─────────────────────┬─────────────────────────────────────────────┐
  │ alignment           │ agreement with majority vote consensus       │
  │ discrimination      │ variance of per-generator scores (spread)   │
  │ consistency         │ stability across evaluator dimensions        │
  │ bias_penalty        │ deviation of positive rate from panel median │
  └─────────────────────┴─────────────────────────────────────────────┘
  composite = alignment + discrimination + consistency - bias_penalty

              alignment  discrimination  consistency  bias_penalty  composite_score  rank
  Claude-3.5     0.8000          0.0070       0.4343        0.0000           2.3129     1
  Gemini-Pro     0.8000          0.0059       0.5946        0.1867           1.5350     2
  GPT-4o         0.7933          0.0041       0.3872        0.0000           1.2667     3
  Gemini-Flash   0.7000          0.0076       0.2835        0.0933           0.5000     4

  ✓ Best judge  : Claude-3.5  (rank 1)
  ✗ Worst judge : Gemini-Flash  (rank 4)

  Why Gemini-Flash ranks last:
    discrimination      : 0.0076
    bias_penalty        : 0.0933
    consistency         : 0.2835
    alignment           : 0.7000

────────────────────────────────────────────────
PART 2 — GENERATOR RANKING  (judge-quality-weighted)
────────────────────────────────────────────────

  Judge weights (derived from composite scores):
    Claude-3.5   : 0.5015  ████████████████████████████████████████
    Gemini-Pro   : 0.2863  ██████████████████████
    GPT-4o       : 0.2121  ████████████████
    Gemini-Flash : 0.0000

  Scores per generator (weighted judge ensemble):

              Halluci.  Adherence  Coherence  mean  weighted  min  rank_mean  rank_min  rank_weighted
  Qwen-2.5     0.8357     0.6643     0.8143  0.771    0.788  0.664      1         1              1
  DS-R1        0.7279     0.6635     0.7099  0.700    0.707  0.664      2         2              2
  QwQ-32B      0.5847     0.7355     0.7463  0.689    0.663  0.585      3         5              3
  DS-R1-Zero   0.6779     0.6173     0.5889  0.628    0.641  0.589      4         4              4
  Kimi-k1.5    0.6143     0.6389     0.6244  0.626    0.623  0.614      5         3              5

  ✓ Best generator (mean score)        : Qwen-2.5
  ✓ Best generator (weakest-link / min): Qwen-2.5

  Best generator per evaluator dimension:
    Hallucination : Qwen-2.5  (0.8357)
    Adherence     : QwQ-32B   (0.7355)
    Coherence     : Qwen-2.5  (0.8143)

================================================================
"""
```
Good, now we have determined the best judge and generator. The final step is to analyze the detailed rationales provided by the best judge for the best generator, to understand its strengths and weaknesses. This can inform future improvements in model design and training.

## Now let's fine tune a smaller instruction-following model on the best generator's outputs, using the best judge as the reward model for RLHF. This should yield a more efficient model that retains the quality of the best generator, but can run faster and cheaper in production.

```bash
python src/train.py 
    --train_data your_training_data.jsonl 
    --output_dir ./fine_tuned_model 
    --model_id Qwen/Qwen3-4B-Instruct-2507 
    --val_split 0.1
```